# Clase 082 — Regresión con árboles

`DecisionTreeRegressor` predice, en cada hoja, la **media del target** de las muestras que
cayeron ahí → la predicción es **escalonada** (*piecewise constant*). Además, el árbol **no
extrapola**: fuera del rango de entrenamiento devuelve la constante de la hoja del extremo.
Contrastamos esto con `LinearRegression`.

Requiere: `numpy`, `matplotlib`, `scikit-learn`.

## 1. Ajuste básico: la escalera

Generamos `y = sin(x) + ruido` y entrenamos árboles de `max_depth=2` y `max_depth=5`. Se ven
las escaleras: más profundidad = más escalones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

RND = 42
rng = np.random.default_rng(RND)
x = np.linspace(-3, 3, 200)
y = np.sin(x) + rng.normal(0, 0.1, size=x.size)
X = x.reshape(-1, 1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, s=12, c="lightgray", label="datos")
for d, color in [(2, "tab:blue"), (5, "tab:red")]:
    m = DecisionTreeRegressor(max_depth=d, random_state=RND).fit(X, y)
    ax.plot(x, m.predict(X), color=color, lw=2, label=f"max_depth={d}")
ax.set_title("Prediccion escalonada del arbol"); ax.legend()
plt.tight_layout(); plt.show()
print("cada hoja devuelve una constante -> funcion piecewise constant")

## 2. MSE en train vs. test

Barremos `max_depth`. Con profundidad alta el MSE de train tiende a 0 (memoriza) mientras el de
test empeora: ahí empieza el sobreajuste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RND)
print(f"{'max_depth':>9} | {'MSE_train':>9} | {'MSE_test':>9}")
for d in [1, 2, 4, 8, None]:
    m = DecisionTreeRegressor(max_depth=d, random_state=RND).fit(X_train, y_train)
    mtr = mean_squared_error(y_train, m.predict(X_train))
    mte = mean_squared_error(y_test, m.predict(X_test))
    print(f"{str(d):>9} | {mtr:>9.4f} | {mte:>9.4f}")

## 3. No extrapolación

Predecimos sobre un rango **más ancho** que el train (`x_new ∈ [-5, 5]`). Fuera del rango de
entrenamiento el árbol devuelve una **meseta plana**: no hay pendiente ni tendencia.

In [ ]:
tree = DecisionTreeRegressor(max_depth=4, random_state=RND).fit(X, y)
x_new = np.linspace(-5, 5, 300)
pred_new = tree.predict(x_new.reshape(-1, 1))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, s=12, c="lightgray", label="datos (train)")
ax.plot(x_new, pred_new, "r-", lw=2, label="prediccion del arbol")
ax.axvspan(-3, 3, color="green", alpha=0.08, label="rango de train")
ax.set_title("El arbol NO extrapola: mesetas planas en los extremos")
ax.legend(); plt.tight_layout(); plt.show()

extremo = tree.predict([[3.0]])[0]
afuera = tree.predict([[5.0]])[0]
print(f"pred en x=3 (borde): {extremo:.4f}  |  pred en x=5 (afuera): {afuera:.4f}")
assert np.isclose(extremo, afuera), "afuera del rango repite la constante del extremo"
print("confirmado: sin extrapolacion")

## 4. Árbol vs. lineal fuera del rango

`LinearRegression` sí extrapola con pendiente (a veces mal, pero con tendencia). El árbol queda
clavado. Ninguno es "mejor" siempre: depende de si la relación real es lineal.

In [ ]:
lin = LinearRegression().fit(X, y)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, s=12, c="lightgray", label="datos (train)")
ax.plot(x_new, tree.predict(x_new.reshape(-1, 1)), "r-", lw=2, label="arbol")
ax.plot(x_new, lin.predict(x_new.reshape(-1, 1)), "b-", lw=2, label="lineal")
ax.axvspan(-3, 3, color="green", alpha=0.08, label="rango de train")
ax.set_title("Extrapolacion: arbol (meseta) vs. lineal (pendiente)")
ax.legend(); plt.tight_layout(); plt.show()
print("el lineal extrapola con tendencia; el arbol se queda en la ultima constante")

## 5. Validación cruzada en un dataset real

Sobre **diabetes** (regresión, sin descargas), comparamos `max_depth=6` contra `max_depth=None`
con `cross_val_score`. El árbol sin límite sobreajusta y generaliza peor.

In [ ]:
data = load_diabetes()
Xd, yd = data.data, data.target
for d in [6, None]:
    scores = cross_val_score(
        DecisionTreeRegressor(max_depth=d, random_state=RND),
        Xd, yd, scoring="neg_mean_squared_error", cv=5, n_jobs=1)
    print(f"max_depth={str(d):>4}: MSE_cv = {-scores.mean():.1f} +/- {scores.std():.1f}")
print("regularizar (max_depth=6) reduce el MSE de CV frente a un arbol sin limite")

## Ejercicios

1. Generá `y = sin(x) + ruido`, entrená árboles de `max_depth=2` y `max_depth=5` y graficá
   ambas predicciones sobre los puntos. Observá las escaleras.
2. Con `train_test_split`, calculá `mean_squared_error` en train y test para
   `max_depth ∈ {1, 2, 4, 8, None}`. ¿Dónde empieza el sobreajuste?
3. Predecí sobre `x_new ∈ [-5, 5]` (más ancho que el train) y graficá las mesetas planas de la
   no-extrapolación.
4. Entrená `LinearRegression` sobre los mismos datos y comparalo con el árbol fuera del rango.
   ¿Cuál extrapola "bien" y por qué?
5. Con `cross_val_score`, compará `DecisionTreeRegressor(max_depth=6)` contra `max_depth=None`
   sobre un dataset real.

## Conclusiones

- Cada hoja predice la **media del target**: la salida es **escalonada** (*piecewise
  constant*), nunca una curva suave.
- El criterio de split por defecto es `squared_error` (MSE): minimiza la varianza ponderada de
  los hijos.
- El árbol **no extrapola**: fuera del rango de train repite la constante del extremo (pendiente
  0). Para tendencias, usá un modelo lineal o diferenciá la serie.
- Sin regularizar, el MSE de train tiende a 0 (memoriza) y el de test se dispara: siempre
  validar con CV.
- Un solo árbol tiene varianza alta; en producción se usa como bloque de **Random Forest** o
  **Gradient Boosting**.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios de la seccion 🧪 **Ejercicios** del README. Cada bloque es autocontenido, se ejecuta **sin internet** y en pocos segundos. Intenta resolver cada ejercicio por tu cuenta antes de mirar la solucion.

### Ejercicio 1 — Ajuste basico
`DecisionTreeRegressor` produce una funcion escalonada (constante por hoja).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
plt.figure(figsize=(6, 4)); plt.scatter(x, y, s=10, c='lightgray')
for d, c in [(2, 'tab:blue'), (5, 'tab:red')]:
    mdl = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X, y)
    o = np.argsort(x); plt.plot(x[o], mdl.predict(X)[o], c=c, lw=2, label=f'depth={d}')
plt.legend(); plt.title('escaleras'); plt.tight_layout(); plt.show()

### Ejercicio 2 — MSE train vs test
Con depth grande el train baja a ~0 pero el test sube (sobreajuste).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
for d in [1, 2, 4, 8, None]:
    mdl = DecisionTreeRegressor(max_depth=d, random_state=42).fit(Xtr, ytr)
    print(f'depth={str(d):4s} MSE train={mean_squared_error(ytr, mdl.predict(Xtr)):.3f} '
          f'test={mean_squared_error(yte, mdl.predict(Xte)):.3f}')

### Ejercicio 3 — No extrapola
Fuera del rango de entrenamiento el arbol repite la hoja del borde: linea plana.

In [ ]:
mdl = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X, y)
x_new = np.linspace(-5, 5, 200); Xn = x_new.reshape(-1, 1)
plt.figure(figsize=(6, 4)); plt.scatter(x, y, s=8, c='lightgray'); plt.plot(x_new, mdl.predict(Xn), 'r-')
plt.axvspan(-3, 3, alpha=0.1, color='green'); plt.title('mesetas planas fuera del rango train')
plt.tight_layout(); plt.show()

### Ejercicio 4 — Arbol vs lineal fuera de rango
El lineal extrapola con pendiente; el arbol se aplana.

In [ ]:
from sklearn.linear_model import LinearRegression
lin = LinearRegression().fit(X, y)
plt.figure(figsize=(6, 4)); plt.scatter(x, y, s=8, c='lightgray')
plt.plot(x_new, mdl.predict(Xn), label='arbol'); plt.plot(x_new, lin.predict(Xn), label='lineal')
plt.legend(); plt.title('extrapolacion'); plt.tight_layout(); plt.show()
print('Ninguno acierta sin(x) fuera de rango, pero el lineal al menos sigue una tendencia.')

### Ejercicio 5 — Dataset real (offline: diabetes en vez de California)
`depth=6` (regularizado) generaliza mejor que un arbol sin limite.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import cross_val_score
Xd, yd = load_diabetes(return_X_y=True)
for d in [6, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=d, random_state=42), Xd, yd,
                        scoring='neg_mean_squared_error', cv=5)
    print(f'depth={str(d):4s} MSE CV={-s.mean():.1f}')